In [15]:
import datetime as dt

import great_tables as gt
import matplotlib.pyplot as plt
import polars as pl
import sf_quant.data as sfd
import sf_quant.performance as sfp
import sf_quant.research as sfr
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.metrics import r2_score
import numpy as np
import scipy.stats as stats


In [2]:
start = dt.date(1996, 1, 1)

#specify in-sample period to be through 2012, since that's when the paper came out?
end = dt.date(2012, 12, 31)
price_filter = 5
signal_name = "qmj_ridge"

#load assets

data = sfd.load_assets(
    start=start,
    end=end,
    columns=[
        "date",
        "barrid",
        "ticker",
        "price",
        "return",
        "daily_volume",
        "market_cap",
        "specific_return",
        "specific_risk",
        "predicted_beta",
        "bid_ask_spread",
    ],
    in_universe=True,
).with_columns(
    pl.col("return").truediv(100),
    pl.col("specific_return").truediv(100),
    pl.col("specific_risk").truediv(100),
)

data.sort('date')

date,barrid,ticker,price,return,daily_volume,market_cap,specific_return,specific_risk,predicted_beta,bid_ask_spread
date,str,str,f64,f64,f64,f64,f64,f64,f64,f64
1996-01-02,"""USAA191""","""AIR""",21.75,-0.011364,15100.0,3.4710825e8,-0.01598,0.226956,0.799488,null
1996-01-02,"""USAA1Y1""","""ADCT""",35.5,-0.027397,274800.0,2.2226e9,-0.02804,0.363066,1.959546,null
1996-01-02,"""USAA2L1""","""AEPI""",22.875,0.039773,4000.0,1.09685625e8,0.03483,0.337045,0.407454,null
1996-01-02,"""USAA311""","""AESC""",23.0,-0.036649,92800.0,1.7181e9,-0.04126,0.274648,0.871196,null
1996-01-02,"""USAA3I1""","""ALO""",26.875,0.028708,39500.0,3.6077e8,0.03612,0.318637,1.156087,null
…,…,…,…,…,…,…,…,…,…,…
2012-12-31,"""USBAHZ1""","""CNSI""",28.53,0.0322,39230.0,6.24941091e8,0.01811,0.303762,1.140558,0.12
2012-12-31,"""USBAIB1""","""RH""",33.73,-0.004721,24569.0,1.2793e9,-0.02694,0.275769,0.998716,0.04
2012-12-31,"""USBAII1""","""OFS""",13.69,0.01936,21711.0,1.3113e8,0.00603,0.213603,1.237185,0.04


In [3]:
# load exposures

exposures = sfd.load_exposures(
    start=start,
    end=end,
    in_universe=True,
    columns=[
        "date",
        "barrid",
        "USSLOWL_EARNQLTY",
        "USSLOWL_VALUE",
        "USSLOWL_PROFIT",
    ],
)

exposures

date,barrid,USSLOWL_EARNQLTY,USSLOWL_VALUE,USSLOWL_PROFIT
date,str,f64,f64,f64
2012-06-29,"""USA0BJ1""",-0.5,-0.243,-1.686
2012-07-02,"""USA0BJ1""",-0.498,-0.238,-1.686
2012-07-03,"""USA0BJ1""",-0.499,-0.25,-1.685
2012-07-05,"""USA0BJ1""",-0.499,-0.231,-1.69
2012-07-06,"""USA0BJ1""",-0.499,-0.225,-1.691
…,…,…,…,…
2012-12-31,"""USBAHZ1""",-2.049,-1.359,-0.161
2012-12-31,"""USBAIB1""",0.139,-0.678,0.894
2012-12-31,"""USBAII1""",-1.044,1.273,-0.81


In [4]:
#initialize quality columns (only relevant 5 from MSCI quality score)

quality_cols = [
    "USSLOWL_PROFIT",
    "USSLOWL_EARNQLTY",
    "USSLOWL_MGMTQLTY",
    "USSLOWL_LEVERAGE",
    "USSLOWL_GROWTH"
]

In [5]:
# load z-scored quality-adjacent exposures

factors = sfd.load_exposures(
    start=start, end=end, in_universe=True, columns=["date", "barrid"] + quality_cols
).with_columns(
    [
        (
            (pl.col(f) - pl.col(f).mean().over("date")) / pl.col(f).std().over("date")
        ).alias(f)
        for f in quality_cols
    ]
)

factors

date,barrid,USSLOWL_PROFIT,USSLOWL_EARNQLTY,USSLOWL_MGMTQLTY,USSLOWL_LEVERAGE,USSLOWL_GROWTH
date,str,f64,f64,f64,f64,f64
2012-06-29,"""USA0BJ1""",-1.412341,-0.041531,-2.523318,0.918733,-0.723896
2012-07-02,"""USA0BJ1""",-1.41325,-0.042086,-2.52273,0.928568,-0.722013
2012-07-03,"""USA0BJ1""",-1.413355,-0.0503,-2.525091,0.928436,-0.720881
2012-07-05,"""USA0BJ1""",-1.414218,-0.049798,-2.520167,0.946213,-0.723777
2012-07-06,"""USA0BJ1""",-1.415084,-0.049662,-2.520883,0.952972,-0.724422
…,…,…,…,…,…,…
2012-12-31,"""USBAHZ1""",0.027212,-1.489162,-0.339065,-0.357549,0.544034
2012-12-31,"""USBAIB1""",1.044456,0.566062,0.379387,-0.371357,1.456358
2012-12-31,"""USBAII1""",-0.598562,-0.545149,-0.307069,-0.861512,-0.123709


In [6]:
#join exposures and factors on crsp

data = data.join(exposures, on=["date", "barrid"], how="left")

data = data.join(factors, on=["date", "barrid"], how="inner")

data

date,barrid,ticker,price,return,daily_volume,market_cap,specific_return,specific_risk,predicted_beta,bid_ask_spread,USSLOWL_EARNQLTY,USSLOWL_VALUE,USSLOWL_PROFIT,USSLOWL_PROFIT_right,USSLOWL_EARNQLTY_right,USSLOWL_MGMTQLTY,USSLOWL_LEVERAGE,USSLOWL_GROWTH
date,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2012-06-29,"""USA0BJ1""","""BLMT""",12.75,0.019185,14532.0,1.1695575e8,0.02415,0.234621,0.284503,0.04,-0.5,-0.243,-1.686,-1.412341,-0.041531,-2.523318,0.918733,-0.723896
2012-07-02,"""USA0BJ1""","""BLMT""",12.9,0.011765,7180.0,1.183317e8,-0.01246,0.233095,0.277438,0.01,-0.498,-0.238,-1.686,-1.41325,-0.042086,-2.52273,0.928568,-0.722013
2012-07-03,"""USA0BJ1""","""BLMT""",12.85,-0.003876,6893.0,1.1787305e8,-0.00226,0.232352,0.272768,0.01,-0.499,-0.25,-1.685,-1.413355,-0.0503,-2.525091,0.928436,-0.720881
2012-07-05,"""USA0BJ1""","""BLMT""",12.7,-0.011673,1129.0,1.164971e8,-0.01158,0.2319,0.276147,0.03,-0.499,-0.231,-1.69,-1.414218,-0.049798,-2.520167,0.946213,-0.723777
2012-07-06,"""USA0BJ1""","""BLMT""",12.67,-0.002362,10010.0,1.1622191e8,-0.01732,0.233326,0.277531,0.03,-0.499,-0.225,-1.691,-1.415084,-0.049662,-2.520883,0.952972,-0.724422
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2012-12-31,"""USBAHZ1""","""CNSI""",28.53,0.0322,39230.0,6.24941091e8,0.01811,0.303762,1.140558,0.12,-2.049,-1.359,-0.161,0.027212,-1.489162,-0.339065,-0.357549,0.544034
2012-12-31,"""USBAIB1""","""RH""",33.73,-0.004721,24569.0,1.2793e9,-0.02694,0.275769,0.998716,0.04,0.139,-0.678,0.894,1.044456,0.566062,0.379387,-0.371357,1.456358
2012-12-31,"""USBAII1""","""OFS""",13.69,0.01936,21711.0,1.3113e8,0.00603,0.213603,1.237185,0.04,-1.044,1.273,-0.81,-0.598562,-0.545149,-0.307069,-0.861512,-0.123709


In [7]:
# Make static QMJ signal, don't shift due to cross-sectional regression on contemporaneous exposures

signals_qmj = data.sort("barrid", "date").with_columns(
    pl.col("USSLOWL_EARNQLTY")
    .add(pl.col("USSLOWL_PROFIT"))
    .truediv(2)
    .alias("qmj")
)

signals_qmj

date,barrid,ticker,price,return,daily_volume,market_cap,specific_return,specific_risk,predicted_beta,bid_ask_spread,USSLOWL_EARNQLTY,USSLOWL_VALUE,USSLOWL_PROFIT,USSLOWL_PROFIT_right,USSLOWL_EARNQLTY_right,USSLOWL_MGMTQLTY,USSLOWL_LEVERAGE,USSLOWL_GROWTH,qmj
date,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2012-06-29,"""USA0BJ1""","""BLMT""",12.75,0.019185,14532.0,1.1695575e8,0.02415,0.234621,0.284503,0.04,-0.5,-0.243,-1.686,-1.412341,-0.041531,-2.523318,0.918733,-0.723896,-1.093
2012-07-02,"""USA0BJ1""","""BLMT""",12.9,0.011765,7180.0,1.183317e8,-0.01246,0.233095,0.277438,0.01,-0.498,-0.238,-1.686,-1.41325,-0.042086,-2.52273,0.928568,-0.722013,-1.092
2012-07-03,"""USA0BJ1""","""BLMT""",12.85,-0.003876,6893.0,1.1787305e8,-0.00226,0.232352,0.272768,0.01,-0.499,-0.25,-1.685,-1.413355,-0.0503,-2.525091,0.928436,-0.720881,-1.092
2012-07-05,"""USA0BJ1""","""BLMT""",12.7,-0.011673,1129.0,1.164971e8,-0.01158,0.2319,0.276147,0.03,-0.499,-0.231,-1.69,-1.414218,-0.049798,-2.520167,0.946213,-0.723777,-1.0945
2012-07-06,"""USA0BJ1""","""BLMT""",12.67,-0.002362,10010.0,1.1622191e8,-0.01732,0.233326,0.277531,0.03,-0.499,-0.225,-1.691,-1.415084,-0.049662,-2.520883,0.952972,-0.724422,-1.095
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2012-12-31,"""USBAHZ1""","""CNSI""",28.53,0.0322,39230.0,6.24941091e8,0.01811,0.303762,1.140558,0.12,-2.049,-1.359,-0.161,0.027212,-1.489162,-0.339065,-0.357549,0.544034,-1.105
2012-12-31,"""USBAIB1""","""RH""",33.73,-0.004721,24569.0,1.2793e9,-0.02694,0.275769,0.998716,0.04,0.139,-0.678,0.894,1.044456,0.566062,0.379387,-0.371357,1.456358,0.5165
2012-12-31,"""USBAII1""","""OFS""",13.69,0.01936,21711.0,1.3113e8,0.00603,0.213603,1.237185,0.04,-1.044,1.273,-0.81,-0.598562,-0.545149,-0.307069,-0.861512,-0.123709,-0.927


In [13]:
# Z-score qmj and ew priors to regress on z-scored exposures

ridge_df = (
    signals_qmj.select(
        ['date','barrid','qmj'] + quality_cols
    ).drop_nulls()
    .with_columns(
        pl.col('qmj')
        .sub(pl.col('qmj').mean().over("date"))
        .truediv(pl.col('qmj').std().over("date"))
        .alias('qmj_z')
    )
    .with_columns(
        pl.sum_horizontal(['USSLOWL_PROFIT','USSLOWL_EARNQLTY','USSLOWL_MGMTQLTY'])
        .sub(pl.sum_horizontal(['USSLOWL_LEVERAGE','USSLOWL_GROWTH']))
        .alias('ew')
    )
    .with_columns(
        pl.col('ew')
        .sub(pl.col('ew').mean().over('date'))
        .truediv(pl.col('ew').std().over('date'))
        .alias('ew_z')
    )
    .sort(['barrid','date'])
)

ridge_df

date,barrid,qmj,USSLOWL_PROFIT,USSLOWL_EARNQLTY,USSLOWL_MGMTQLTY,USSLOWL_LEVERAGE,USSLOWL_GROWTH,qmj_z,ew,ew_z
date,str,f64,f64,f64,f64,f64,f64,f64,f64,f64
2012-06-29,"""USA0BJ1""",-1.093,-1.686,-0.5,-2.523318,0.918733,-0.723896,-0.895545,-4.904155,-1.709277
2012-07-02,"""USA0BJ1""",-1.092,-1.686,-0.498,-2.52273,0.928568,-0.722013,-0.895907,-4.913285,-1.714303
2012-07-03,"""USA0BJ1""",-1.092,-1.685,-0.499,-2.525091,0.928436,-0.720881,-0.901831,-4.916647,-1.723251
2012-07-05,"""USA0BJ1""",-1.0945,-1.69,-0.499,-2.520167,0.946213,-0.723777,-0.902015,-4.931603,-1.726835
2012-07-06,"""USA0BJ1""",-1.095,-1.691,-0.499,-2.520883,0.952972,-0.724422,-0.902444,-4.939434,-1.728827
…,…,…,…,…,…,…,…,…,…,…
2012-12-31,"""USBAHZ1""",-1.105,-0.161,-2.049,-0.339065,-0.357549,0.544034,-0.922969,-2.73555,-0.839519
2012-12-31,"""USBAIB1""",0.5165,0.894,0.139,0.379387,-0.371357,1.456358,0.999259,0.327386,0.395127
2012-12-31,"""USBAII1""",-0.927,-0.81,-1.044,-0.307069,-0.861512,-0.123709,-0.711957,-1.175848,-0.210815


In [9]:
# set prior vector weights

# Null prior (standard ridge formulation)
p0 = np.zeros(len(quality_cols))

# QMJ prior
p1 = np.zeros(len(quality_cols))
p1[quality_cols.index("USSLOWL_PROFIT")] = 0.5
p1[quality_cols.index("USSLOWL_EARNQLTY")] = 0.5

# Equal weight prior according to MSCI construction
p2 = np.array([1,1,1,-1,-1]) / 5

In [10]:
# ridge regression

def ridge_with_prior(X,y,w0,lam):
    y_tilde = y - X @ w0
    model = Ridge(alpha=lam, fit_intercept=False)
    model.fit(X, y_tilde)
    return model.coef_ + w0

In [17]:
methods = {
    "ols_qmj":             {"target": "qmj_z", "w0": p0, "lam": 0.0}, #purely data (OLS), static 50/50 target
    "ols_ew":              {"target": "ew_z", "w0": p0, "lam": 0.0}, #purely data (OLS), MSCI quality score target
    "ridge_ew_loose":      {"target": "ew_z", "w0": p2, "lam": 1.0}, #MSCI target and prior, minimal penalty (loose)
    "ridge_ew_tight":      {"target": "ew_z", "w0": p2, "lam": 100.0}, #MSCI target and prior, large penalty (tight)
}

weight_records = {name: [] for name in methods}
r2_records = {name: [] for name in methods}


for (date,), df_date in ridge_df.group_by("date", maintain_order=True):
    X_t = df_date.select(quality_cols).to_numpy()

    for name, cfg in methods.items():
        y_t = df_date.select(cfg["target"]).to_numpy().flatten()

        if cfg["lam"] == 0.0:
            model = LinearRegression(fit_intercept=False)
            model.fit(X_t, y_t)
            w_t = model.coef_
        else:
            w_t = ridge_with_prior(X_t, y_t, cfg["w0"], cfg["lam"])

        y_hat = X_t @ w_t
        ss_res = np.sum((y_t - y_hat) ** 2)
        ss_tot = np.sum(y_t ** 2)
        r2 = 1 - ss_res / ss_tot

        weight_records[name].append(w_t)
        r2_records[name].append(r2)



static_weights = {
    name: np.mean(ws, axis=0)
    for name, ws in weight_records.items()
}

mean_r2 = {
    name: np.mean(rs) 
    for name, rs in r2_records.items()
}

# Claude-assisted formatting to understand outputs in a more organized fashion

print(f"\n{'Method':<22} {'Factor':<25} {'Raw W':>8}  {'Norm W':>8}  {'Mean R²':>8}")
print("─" * 80)

for name, w in static_weights.items():
    w_norm = w / np.sum(w)
    r2 = mean_r2[name]
    for i, factor in enumerate(quality_cols):
        label = name if i == 0 else ""
        r2_str = f"{r2:.4f}" if i == 0 else ""
        print(f"{label:<22} {factor:<25} {w[i]:>8.4f}  {w_norm[i]:>8.4f}  {r2_str:>8}")
    print()


Method                 Factor                       Raw W    Norm W   Mean R²
────────────────────────────────────────────────────────────────────────────────
ols_qmj                USSLOWL_PROFIT              0.6016    0.5571    0.9017
                       USSLOWL_EARNQLTY            0.4816    0.4460          
                       USSLOWL_MGMTQLTY            0.0257    0.0238          
                       USSLOWL_LEVERAGE           -0.0191   -0.0177          
                       USSLOWL_GROWTH             -0.0099   -0.0092          

ols_ew                 USSLOWL_PROFIT              0.3812    1.2695    0.9606
                       USSLOWL_EARNQLTY            0.3055    1.0174          
                       USSLOWL_MGMTQLTY            0.4005    1.3337          
                       USSLOWL_LEVERAGE           -0.3963   -1.3200          
                       USSLOWL_GROWTH             -0.3905   -1.3006          

ridge_ew_loose         USSLOWL_PROFIT              0.3811 

In [20]:
for name, ws in weight_records.items():
    ws_arr = np.array(ws)  # (T, 5)
    means = ws_arr.mean(axis=0)
    stds = ws_arr.std(axis=0, ddof=1)
    cv = stds / np.abs(means)  # coefficient of variation

    print(f"\n{name}")
    print(f"{'Factor':<25} {'Mean':>8}  {'Std':>8}  {'CV':>8}")
    for i, f in enumerate(quality_cols):
        print(f"  {f:<25} {means[i]:>8.4f}  {stds[i]:>8.4f}  {cv[i]:>8.4f}")


ols_qmj
Factor                        Mean       Std        CV
  USSLOWL_PROFIT              0.6016    0.0401    0.0667
  USSLOWL_EARNQLTY            0.4816    0.0463    0.0962
  USSLOWL_MGMTQLTY            0.0257    0.0149    0.5788
  USSLOWL_LEVERAGE           -0.0191    0.0151    0.7888
  USSLOWL_GROWTH             -0.0099    0.0129    1.3005

ols_ew
Factor                        Mean       Std        CV
  USSLOWL_PROFIT              0.3812    0.0297    0.0779
  USSLOWL_EARNQLTY            0.3055    0.0349    0.1141
  USSLOWL_MGMTQLTY            0.4005    0.0243    0.0606
  USSLOWL_LEVERAGE           -0.3963    0.0262    0.0661
  USSLOWL_GROWTH             -0.3905    0.0257    0.0659

ridge_ew_loose
Factor                        Mean       Std        CV
  USSLOWL_PROFIT              0.3811    0.0297    0.0778
  USSLOWL_EARNQLTY            0.3055    0.0349    0.1141
  USSLOWL_MGMTQLTY            0.4004    0.0242    0.0606
  USSLOWL_LEVERAGE           -0.3963    0.0262    0.0660
  US